<a href="https://colab.research.google.com/github/UOS-COMP6252/public/blob/main/lecture6/performance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import getpass
import os
try:
  import comet_ml
except ModuleNotFoundError:
  %pip install comet_ml
  import comet_ml
comet_api_key=os.environ.get("COMET_API_KEY")
if comet_api_key is None:
  comet_api_key=getpass.getpass("Enter key")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision as vision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torch.optim import SGD,Adam

### Dropout


In [ ]:
class DP(nn.Module):
    def __init__(self,prob=0.5):
        super().__init__()
        self.dp=nn.Dropout(1-prob)
    def forward(self,x):
        return self.dp(x)

### Training vs test/validate
- Implementation in PyTorch opts for 
1. $\frac{1}{p}$ during training
2. Identity during testing/validation

In [ ]:
x=torch.tensor([[1,2],[3,4]],dtype=torch.float32).unsqueeze(0)
# keep prob=0.9
dp=DP(prob=0.8)
dp.train()
print(dp(x).squeeze())
dp.eval()
dp(x).squeeze()

## Limiting values

If we perform a large number of trials $1-p$ values are set to zero

In [ ]:
dp.train()
count=0.
n=50
for i in range(n):
    count+=(dp(x)==torch.zeros(2,2)).sum().item()

print(count/4/n)

### Batch normalization     
Given a mini-batch of tensors $x_{ci}$ of dimension (S,C,H,W) where $c$ is the channel index and $i$ collectively refers to all other dimensions. 

Let $N=S\times H\times W$. Batch normalization computes the mean and variance of the batch (per channel) according to
  $$
    \begin{align*}
    \mu_c&=\frac{1}{N}\sum_{i=1}^N x_{ci}\\
    \sigma^2_c&=\frac{1}{N}\sum_{i=1}^N \left(x_{ci}-\mu_c\right)^2
    \end{align*}
$$

The normalized inputs are computed as follows:
$$
\begin{align*}
\hat{x}_{ci}=\frac{x_{ic}-\mu_c}{\sqrt{\sigma^2_c+\epsilon}}
\end{align*}
$$

Therefore, for each channel, the $\hat{x}_{ci}$ have zero mean and unit variance. The output of the batch normalization layer is given by
$$
\begin{align*}
y_{ic}=\gamma \hat{x}_{ic}+\beta
\end{align*}
$$
Where $\gamma$ and $\beta$ are **learnable** parameters.

#### Example
- For simplicity we consider a  tensor with a two channels
- Recall that batch normalization is done for each channel independently
- In the example below we create an arbitrary tensor ```a```  of size ```(2,1,22)```
- It represents two samples, each with a single channel representing a 2x2 tensor.


In [ ]:
x1=torch.tensor([[1,223],[3,444]],dtype=torch.float32).unsqueeze(0)
x2=torch.tensor([[12,11],[41,32]],dtype=torch.float32).unsqueeze(0)
x=torch.vstack([x1,x2])
y1=torch.tensor([[5,6],[7,18]],dtype=torch.float32).unsqueeze(0)
y2=torch.tensor([[16,5],[8,1777]],dtype=torch.float32).unsqueeze(0)
y=torch.vstack([y1,y2])
a=torch.stack([x,y])
print("a size={}".format(list(a.size())))
print("a's first channel\n{}".format(a[:,0,:,:].numpy()))
print("a's second channel\n{}".format(a[:,1,:,:].numpy()))

##### Manual Computation vs Normalization Layer


In [ ]:
# using PyTorch BatchNorm2d
class BN(nn.Module):
    def __init__(self):
        super().__init__()
        self.norm=nn.BatchNorm2d(num_features=2)
    def forward(self,x):
        return self.norm(x)

In [ ]:
# Manually compute the mean and average
# include all values in dimensions 0,2,3
# i.e. compute over the batch, height and width dimensions (skipping the channel dimension)
var=a.var([0,2,3],unbiased=False)
mean=a.mean([0,2,3])
anorm=torch.ones_like(a)
print("mean={} and variance={}".format(mean,var))
for i in range(2):
    anorm[:,i,:,:]=(a[:,i,:,:]-mean[i])/torch.sqrt(var[i])

- During the training phase, every time the mean and variance are computed
- They are accumulated into a running mean and variance according to the formula
$$
x_{new}=0.9*x_{old}+0.1*x
$$
where $x$ is the observed value of mean/variance.

Initially the mean is 0 and variance is 1

In [ ]:
#bn=BN()    
bn=nn.BatchNorm2d(num_features=2)
bn.train()
print("mean ={}, variance ={}".format(bn.running_mean,bn.running_var))

Recall the output of the batch norm layer is 
$$
\begin{align*}
y_{ic}=\gamma \hat{x}_{ic}+\beta
\end{align*}
$$
$\gamma$ and $\beta$ are  learnable parameters initially set to 1 and 0 respectively as shown below.

How would one know that $\gamma$ and $\beta$ are **learnable** parameters?

Why do they have 2 values?

In [ ]:
p=bn.parameters()
gamma=next(p)
beta=next(p)
print(gamma)
print(beta)
try:
    np=next(p)
except StopIteration:
    print("No more parameters")

In [ ]:
c=bn(a)
for i in range(2):
    print("Manual channel {}".format(i))
    print(anorm[:,i,:,:].numpy())
    print("\n BN channel {}\n".format(i))
    print(c[:,i,:,:].detach().numpy())
    print("---------------------")

#### Testing/validation phase
- During training a running average of the mean and **unbiased** variance is computed
- initially the mean is set to 0 and variance to 1
- For each iteration (step) the mean  and variance are computed and the running average of those are updated using the formula

$$
 running\_mean=0.9\times running\_mean+0.1\times mean\\ 
 running\_var=0.9\times running\_var+0.1\times var 
 $$
- When ```model.eval()``` is used the ```batchNorm2d``` layers uses the running averages

In [ ]:
ave_mean=0.9*0+0.1*a.mean([0,2,3])
ave_var=0.9*1+0.1*a.var([0,2,3],unbiased=True)
print("From BN layer")
print("running_mean={}, running_var={}".format(bn.running_mean,bn.running_var))
print("---------------------")
print("From manual computation")
print("running_mean={},running_var={}".format(ave_mean,ave_var))

#### How is the ouput computed during test/validate

Let $x_i$ be the input to batch normalisation layer $BN$ during inference (i.e. test/validate) for channel $i$

Then $BN(x_i)=\frac{x_i-\mu_i}{\sigma_i}$

Where $\mu_i$ and $\sigma_i$ are the running averages for channel $i$ computed during training

In [ ]:
#run bn on input a and compare with manual computation
bn.eval()
for i in range(2):
    anorm[:,i,:,:]=(a[:,i,:,:]-bn.running_mean[i])/torch.sqrt(bn.running_var[i])
print(anorm[:,0,:,:])
print(bn(a)[:,0,:,:])

### Convolution Network for CIFAR10

#### Setup and global parameters

In [ ]:
# to ensure reproducibility
seed=9 
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True
# use/not use batch normalization


#### Datasets

In [ ]:
batch_size=64
transform = transforms.ToTensor()
dataset_train=vision.datasets.CIFAR10(".",download=True,train=True,transform=transform)
dataset_test=vision.datasets.CIFAR10(".",download=True,train=False,transform=transform)
loader_train=DataLoader(dataset_train,batch_size=batch_size,shuffle=True,num_workers=2)
loader_test=DataLoader(dataset_test,batch_size=batch_size,shuffle=False)

### Network architecture

In [ ]:
class Net(nn.Module):
  def __init__(self,norm_layers=True):
    super().__init__()
    self.norm_layers=norm_layers
    self.norm1=nn.BatchNorm2d(32)
    self.norm2=nn.BatchNorm2d(32)
    self.norm3=nn.BatchNorm2d(64)
    self.norm4=nn.BatchNorm2d(64)
    self.relu=nn.ReLU()
    # input is (*,3,32,32)
    self.conv1=nn.Conv2d(in_channels=3,out_channels=32,kernel_size=3)
    # input is (*,32,30,30)
    self.conv2=nn.Conv2d(in_channels=32,out_channels=32,kernel_size=3)
    self.conv2a=nn.Conv2d(in_channels=32,out_channels=32,kernel_size=3)
    # input is (*,32,28,28)
    self.pool1=nn.MaxPool2d(kernel_size=(2,2))
    # input is (*,32,14,14)
    self.conv3=nn.Conv2d(in_channels=32,out_channels=64,kernel_size=3)
    # input is (*,64,12,12)
    self.conv4=nn.Conv2d(in_channels=64,out_channels=64,kernel_size=3)
    self.conv4a=nn.Conv2d(in_channels=64,out_channels=64,kernel_size=3)
    # input is (*,64,10,10)
    self.pool2=nn.MaxPool2d(kernel_size=(2,2))
    # input is (*,64,5,5)
    self.flatten=nn.Flatten()
    # input is (*,64x5x5)
    self.fc1=nn.Linear(in_features=5*5*64,out_features=128)
    self.fc2=nn.Linear(in_features=128,out_features=10)

  def forward(self,x):
    x=self.conv1(x)
    if self.norm_layers:
      x=self.norm1(x)
    x=self.relu(x)
    x=self.conv2(x)
    if self.norm_layers:
      x=self.norm2(x)
    x=self.relu(x)
    x=self.pool1(x)
    
    x=self.conv3(x)
    if self.norm_layers:
      x=self.norm3(x)
    x=self.relu(x)
    x=self.conv4(x)
    if self.norm_layers:
      x=self.norm4(x)
    x=self.relu(x)
    x=self.pool2(x)
    
    x=self.flatten(x)
    x=self.fc1(x)
    x=self.relu(x)
    x=self.fc2(x)
    return x
    

In [ ]:
try:
    from torchmetrics import ConfusionMatrix
    from torchmetrics.classification import MulticlassAccuracy
except ModuleNotFoundError:
    %pip install torchmetrics
    from torchmetrics import ConfusionMatrix
    from torchmetrics.classification import MulticlassAccuracy


#### Accuracy/loss computation

In [ ]:
def accuracy(model,batch,loss_fn):
    accuracy=MulticlassAccuracy(10).cuda()
    imgs,labels=batch
    imgs,labels=imgs.cuda(),labels.cuda()
    outputs=model(imgs)
    _,pred=torch.max(outputs,dim=1)
    acc=torch.sum(pred==labels).item()
    loss=loss_fn(outputs,labels)
    accuracy.update(pred,labels)
    return loss,torch.tensor(acc/len(labels)),accuracy.compute().item()

@torch.no_grad() 
def evaluate(model,loader,loss_fn):
    model.eval()
    # crit is a list of pairs of tensors
    crit=[accuracy(model,batch,loss_fn) for batch in loader]
    crit=torch.tensor(crit)
    m=crit.mean(dim=0)
    loss=m[0]
    acc=m[1]
    tm_accuracy=m[2]
    return loss,acc,tm_accuracy

#### Simple early stopping code

In [ ]:
class EarlyStopping():
    def __init__(self,patience=4,tolerance=0):
        self.patience=patience
        self.tolerance=tolerance
        self.min_loss=float('inf')
        self.count=0
    def __call__(self,loss):
        if loss<self.min_loss:
            self.count=0
            self.min_loss=loss
            return False
        elif loss>self.min_loss+self.tolerance:
            self.count+=1
            if self.count>self.patience:
                return True
        return False


#### Main loop

In [ ]:
lr=0.001
use_BN=True
epochs=11
model=Net(norm_layers=use_BN).cuda()
optimizer=Adam(model.parameters())
#optimizer=SGD(model.parameters(),lr=lr)
loss_fn=nn.CrossEntropyLoss()

In [ ]:
experiment = comet_ml.Experiment(api_key=comet_api_key,workspace="COMP6252",project_name="batchnorm",auto_metric_logging=False, auto_output_logging=False)

In [ ]:
experiment.log_parameters({'lr':lr,'batch_size':batch_size,'epochs':epochs})
trigger=True
es=EarlyStopping()
from tqdm import tqdm
for epoch in range(epochs):
  loop=tqdm(loader_train)
  loop.set_description(f"Epoch [{epoch+1}/{epochs}]")
  epoch_loss=0.
  model.train()
  for (imgs,labels) in loop:
    optimizer.zero_grad()
    imgs,labels=imgs.cuda(),labels.cuda()
    outputs=model(imgs)
    loss=loss_fn(outputs,labels)
    loss.backward()
    optimizer.step()
    epoch_loss=0.9*epoch_loss+0.1*loss.item()
    loop.set_postfix(loss=epoch_loss)
   
  t_loss,_,t_acc=evaluate(model,loader_train,loss_fn)
  v_loss,_,v_acc=evaluate(model,loader_test,loss_fn)
  experiment.log_metrics({'loss':epoch_loss,'val_loss':v_loss,'train_acc':t_acc,'val_accuracy':v_acc}, epoch=epoch)
  
  if es(v_loss) and trigger:
  #   break
     print("At epoch={} we should stop. Validation accuracy={}".format(epoch,v_acc))
     trigger=False


#### Metrics

In [ ]:

conmat=ConfusionMatrix(task='multiclass',num_classes=10)
conmat=conmat.cuda()

In [ ]:
total=0
correct=0
accuracy=MulticlassAccuracy(10).cuda()
for imgs,labels in loader_test:
  imgs,labels=imgs.cuda(),labels.cuda()
  outputs=model(imgs)
  # the second return value is the index of the max i.e. argmax
  _,predicted=torch.max(outputs.data,1)
  accuracy.update(predicted,labels)
  correct+=(predicted==labels).sum()
  total+=labels.size()[0]
  conmat.update(predicted,labels)
x=conmat.compute().cpu().numpy()
test_accuracy=accuracy.compute().cpu().numpy().item()
experiment.log_metrics({"test_accuracy":test_accuracy})
experiment.log_confusion_matrix(matrix=x,labels=dataset_train.classes)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sb

plt.figure(figsize=(10,7))
sb.heatmap(x,xticklabels=dataset_train.classes,yticklabels=dataset_train.classes,annot=True,fmt=".0f")

- The rows are the actual images and the columns are the prediction (How can you check?)
- While the prediction accuracy is good albeit not impressive
- From the confusion matrix we find justifications for the inaccuracies
- For example
    - most of the incorrect classifications of automobiles were classified as trucks
    - most of the incorrect classifications of cats/dogs were classified as dogs/cats
    

In [ ]:
experiment.end()